# Agent Walkthrough

Uses the standard `openai` SDK pointed at **OpenTyphoon's OpenAI-compatible endpoint** — same code shape as OpenAI direct, OpenRouter, Together, Groq — swap `base_url` to switch providers.

**Sections:**
1. **Chat Completions** — the bread and butter
2. **Custom Function Calling** — your own Python functions as tools
3. **Web Search via EXA** — grounded search with citations
4. **Python REPL** — sandboxed Python via `langchain` (drop-in for code-interpreter)
5. **Generic Agent Loop** — tiny reusable agent harness
6. **Agentic Search — unstructured data** (file-based RAG over product docs)
7. **Agentic Search — structured data** (CSV) — *two versions*: custom search tool vs. `grep_csv`
8. **Text Editing Agent** — list / read / edit tools
9. **Deep Research Agent** — orchestrator with parallel EXA sub-agents + validator

**You need two keys** (set in your environment or drop into the cell below):
- `TYPHOON_API_KEY` — from [opentyphoon.ai](https://docs.opentyphoon.ai/)
- `EXA_API_KEY` — from [exa.ai](https://exa.ai)


In [ ]:
!pip install -q openai requests langchain-experimental

## Setup — keys and client

In [ ]:
from openai import OpenAI
from google.colab import userdata
import json

MODEL = 'typhoon-v2.5-30b-a3b-instruct'
MAX_TOKENS = 8192  # OpenTyphoon counts prompt + completion; keep generous
client = OpenAI(
    api_key=userdata.get('typhoon'),
    base_url='https://api.opentyphoon.ai/v1',
)

## Verify the connection works

In [ ]:
ping = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Say hello in 5 words.'}],
    max_tokens=MAX_TOKENS,
)
print(ping.choices[0].message.content)


Hello! How can I help you today?


## 1. Chat Completions

One-shot request, one-shot answer. `messages` is a list of `{role, content}` dicts. `role` is one of `system`, `user`, `assistant`, `tool`.

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'You are a concise assistant.'},
        {'role': 'user',   'content': 'Name three Thai street food dishes.'},
    ],
    temperature=0.7,
    max_tokens=MAX_TOKENS,
)
print(resp.choices[0].message.content)


1. Pad Thai  
2. Som Tum (Papaya Salad)  
3. Moo Ping (Grilled Pork Skewers)


## 2. Custom Function Calling

Define a Python function, describe it as a tool schema, and the model will decide when to call it. You execute the call locally and feed the result back.

In [ ]:
import json

def get_exchange_rate(from_currency: str, to_currency: str) -> str:
    rates = {'USD': 35.5, 'EUR': 38.2, 'JPY': 0.24, 'THB': 1.0}
    rate = rates[from_currency] / rates[to_currency]
    return f'1 {from_currency} = {rate:.4f} {to_currency}'

tools = [{
    'type': 'function',
    'function': {
        'name': 'get_exchange_rate',
        'description': 'Get exchange rate between two currencies.',
        'parameters': {
            'type': 'object',
            'properties': {
                'from_currency': {'type': 'string'},
                'to_currency':   {'type': 'string'},
            },
            'required': ['from_currency', 'to_currency'],
        },
    },
}]

messages = [{'role': 'user', 'content': 'How much is 100 USD in THB?'}]

resp = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, max_tokens=MAX_TOKENS)
msg = resp.choices[0].message
messages.append(msg)

for call in (msg.tool_calls or []):
    args = json.loads(call.function.arguments)
    result = get_exchange_rate(**args)
    messages.append({'role': 'tool', 'tool_call_id': call.id, 'content': result})

final = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=MAX_TOKENS)
print(final.choices[0].message.content)


100 USD is equal to 3,550 THB (Thai Baht).


In [ ]:
resp

ChatCompletion(id='chatcmpl-ba9fc0bec14a0a5f', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-8d5db378a86cbc0e', function=Function(arguments='{"from_currency": "USD", "to_currency": "THB"}', name='get_exchange_rate'), type='function')], provider_specific_fields={'refusal': None, 'reasoning': None}))], created=1777362023, model='typhoon-v2.5-30b-a3b-instruct', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=30, prompt_tokens=176, total_tokens=206, completion_tokens_details=None, prompt_tokens_details=None), duration=0.32036, is_harm=None, prompt_logprobs=None, prompt_token_ids=None, kv_transfer_params=None)

## 3. Web Search via EXA

EXA is a neural web search API. We wrap it as a plain Python function so the model (or your code) can call it. Free tier is generous for demos.

Endpoint: `POST https://api.exa.ai/search` with headers `x-api-key: <EXA_API_KEY>` and a JSON body.

In [ ]:
import requests

def exa_search(
    query: str,
    num_results: int = 5,
    include_domains: list[str] | None = None,
    start_published_date: str | None = None,  # 'YYYY-MM-DD'
) -> str:
    """Call EXA /search and return a compact text summary of the results."""
    payload: dict = {
        'query': query,
        'numResults': num_results,
        'contents': {'text': {'maxCharacters': 800}},
    }
    if include_domains:
        payload['includeDomains'] = include_domains
    if start_published_date:
        payload['startPublishedDate'] = start_published_date

    r = requests.post(
        'https://api.exa.ai/search',
        headers={'x-api-key': userdata.get('EXA_API_KEY'), 'content-type': 'application/json'},
        json=payload,
        timeout=30,
    )
    r.raise_for_status()
    data = r.json()

    lines = []
    for i, res in enumerate(data.get('results', []), 1):
        title = res.get('title', '(no title)')
        url   = res.get('url', '')
        text  = (res.get('text') or '')[:600].replace('\n', ' ')
        lines.append(f'[{i}] {title}\n    {url}\n    {text}')
    return '\n\n'.join(lines) if lines else '(no results)'

# Smoke test
print(exa_search('latest Python stable version 2026', num_results=10)[:800])


[1] Download Python - Python.org
    https://www.python.org/downloads/
    Download Python | Python.org  Notice: This page displays a fallback because interactive scripts did not run. Possible causes include disabled JavaScript or failure to load scripts or stylesheets.  Join us in Long Beach, CA starting May 13, 2026. Grab your ticket and discounted hotel today before they’re gone! REGISTER FOR PYCON US!  ## Active Python releases  For more information visit the Python Developer's Guide.  Python version Maintenance status First released End of support Release schedule  ## Looking for a specific release?  Python releases by version number:  Release version Release da

[2] Python 3.15.0a8, 3.14.4 and 3.13.13 are out!
    https://blog.python.org/2026/04/python-3150a8-3144-31313/
    Python 3


### 3a — Basic search

In [ ]:
query = 'What is the latest stable Python version released? Cite sources.'
context = exa_search('latest stable Python version release 2026', num_results=4)

r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'Answer using the provided web search results. Cite sources with URLs.'},
        {'role': 'user',   'content': f'Web results:\n{context}\n\nQuestion: {query}'},
    ],
    max_tokens=MAX_TOKENS,
)
print(r.choices[0].message.content)


As of the provided information, the latest stable Python version released is **Python 3.14.4**, which was released on April 7, 2026. This is described as the fourth maintenance release of Python 3.14, containing numerous bug fixes, build improvements, and documentation updates.

Python 3.15.0a8 is also mentioned, but it is labeled as an "early developer preview" and the final planned alpha release for Python 3.15, meaning it is not yet stable.

Sources:
- [Python Release Python 3.14.4 | Python.org](https://www.python.org/downloads/latest)
- [Python 3.15.0a8, 3.14.4 and 3.13.13 are out! | Python Insider](https://blog.python.org/2026/04/python-3150a8-3144-31313/)


### 3b — Recent news (date-filtered)

EXA supports `startPublishedDate` for recency filtering.

In [ ]:
context = exa_search(
    'Thailand tourism trending destinations',
    num_results=5,
    start_published_date='2026-01-01',
)
r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'Summarize trending Thai tourist destinations based on the results.'},
        {'role': 'user',   'content': context},
    ],
    max_tokens=MAX_TOKENS,
)
print(r.choices[0].message.content)


### 3c — Domain-restricted

In [ ]:
context = exa_search(
    'PEP 8 line length recommendation',
    num_results=3,
    include_domains=['docs.python.org', 'peps.python.org'],
)
r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'user', 'content': f'Based on these Python docs, what does PEP 8 say about line length?\n\n{context}'},
    ],
    max_tokens=MAX_TOKENS,
)
print(r.choices[0].message.content)


### 3d — Web search as a tool (function calling)

Sections 3a–3c call `exa_search()` directly in your code. Here the model **decides when and what to search** — same pattern as section 2's `get_exchange_rate` tool.

In [ ]:
import json
from datetime import date

# --- Tool description (tells the model what it can call) ---
WEB_SEARCH_TOOL = {
    'type': 'function',
    'function': {
        'name': 'exa_search',
        'description': (
            'Search the web using EXA and return results with titles, URLs, and text snippets. '
            'Use this when the user asks a question that requires up-to-date or real-world information.'
        ),
        'parameters': {
            'type': 'object',
            'properties': {
                'query':       {'type': 'string', 'description': 'The search query.'},
                'num_results': {'type': 'integer', 'description': 'Number of results (default 5).'},
            },
            'required': ['query'],
        },
    },
}

# --- Ask a question — the model decides whether to search ---
messages = [
    {'role': 'system', 'content': f"""You are a helpful assistant with web search. Cite URLs when you use search results.
                                      Current date is {date.today().strftime("%Y-%m-%d")}."""},
    {'role': 'user',   'content': 'Who won the latest Nobel Prize in Physics and for what?'},
]

resp = client.chat.completions.create(model=MODEL, messages=messages, tools=[WEB_SEARCH_TOOL], max_tokens=MAX_TOKENS)
msg = resp.choices[0].message
messages.append(msg)

# --- Execute any tool calls the model made ---
for call in (msg.tool_calls or []):
    args = json.loads(call.function.arguments)
    print(f'Model called: exa_search({args})')
    result = exa_search(**args)
    messages.append({'role': 'tool', 'tool_call_id': call.id, 'content': result})

# --- Second pass: model reads the results and answers ---
if msg.tool_calls:
    final = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=MAX_TOKENS)
    print('\n' + final.choices[0].message.content)
else:
    print(msg.content)

Model called: exa_search({'query': 'latest Nobel Prize in Physics winner and reason 2025', 'num_results': 3})

The winners of the 2025 Nobel Prize in Physics are **John Clarke**, **Michel H. Devoret**, and **John M. Martinis**.

They were awarded the prize for their groundbreaking work on **the discovery of macroscopic quantum mechanical tunnelling and energy quantization in electric circuits**. This research laid the foundation for the development of superconducting qubits, which are now central to the field of quantum computing.

Their achievements have enabled the manipulation of quantum states at a macroscopic scale, bringing quantum physics into practical applications such as quantum computers. The prize was announced on 7 October 2025 by the Royal Swedish Academy of Sciences.

Sources:
- [Physics World](https://physicsworld.com/a/john-clarke-michel-devoret-and-john-martinis-win-the-2025-nobel-prize-for-physics/)
- [NobelPrize.org](https://www.nobelprize.org/prizes/physics/2025/pr

## 4. Python REPL (Code Execution)

A simple sandboxed Python REPL via `langchain_experimental`. Gives the model the ability to run Python when it needs to compute, analyze data, or plot.

The REPL keeps a persistent namespace across `.run()` calls — so variables you define in one call are visible in the next.

In [ ]:
from langchain_experimental.utilities.python import PythonREPL

repl = PythonREPL()

def python_repl(code: str) -> str:
    """Execute Python code in the shared REPL. Returns captured stdout,
    or if stdout is empty, evaluates the last expression (like a real REPL)."""
    out = repl.run(code)
    if out.strip():
        return out[:4000]
    # Fallback: try evaluating the last line as an expression
    lines = [l for l in code.strip().splitlines() if l.strip() and not l.strip().startswith('#')]
    if lines:
        try:
            result = repl.run(f'print(repr({lines[-1]}))')
            if result.strip():
                return result[:4000]
        except Exception:
            pass
    return '(no output — use print() to see results)'

PYTHON_REPL_TOOL = {
    'type': 'function',
    'function': {
        'name': 'python_repl',
        'description': (
            'Execute Python code in a sandboxed REPL. '
            'Persistent state between calls (variables stay alive). '
            'Use `print(...)` to show output. '
            'Handy for math, data analysis (`pandas`), string processing.'
        ),
        'parameters': {
            'type': 'object',
            'properties': {'code': {'type': 'string', 'description': 'Python code to execute.'}},
            'required': ['code'],
        },
    },
}

# Quick sanity check
print(python_repl('import math\nmath.factorial(10)'))  # no print(), should still show result

3628800



### 4a — Pure compute (factorial + trailing zeros)

In [ ]:
messages = [
    {'role': 'user', 'content': (
        'Compute 10! Then count how many trailing zeros it has. '
        'Use the python_repl tool — do not guess.'
    )},
]
resp = client.chat.completions.create(model=MODEL, messages=messages, tools=[PYTHON_REPL_TOOL], max_tokens=MAX_TOKENS)
msg = resp.choices[0].message
messages.append(msg)

for call in (msg.tool_calls or []):
    print(f'→ tool: python_repl({json.loads(call.function.arguments)["code"]!r})')
    result = python_repl(**json.loads(call.function.arguments))
    print(f'← {result}\n')
    messages.append({'role': 'tool', 'tool_call_id': call.id, 'content': result})

final = client.chat.completions.create(model=MODEL, messages=messages, tools=[PYTHON_REPL_TOOL], max_tokens=MAX_TOKENS)
print(final.choices[0].message.content)


→ tool: python_repl("import math\nfactorial_10 = math.factorial(10)\ntrailing_zeros = len(str(factorial_10)) - len(str(factorial_10).rstrip('0'))\n(factorial_10, trailing_zeros)")
← (3628800, 2)


The value of $10!$ is **3,628,800**, and it has **2 trailing zeros**.


### 4b — ASCII art (Christmas tree)

In [ ]:
messages = [
    {'role': 'user', 'content': (
        'Generate a 10-row centered ASCII Christmas tree (stars). '
        'Use the python_repl tool to print it.'
    )},
]
resp = client.chat.completions.create(model=MODEL, messages=messages, tools=[PYTHON_REPL_TOOL], max_tokens=MAX_TOKENS)
msg = resp.choices[0].message
messages.append(msg)

for call in (msg.tool_calls or []):
    result = python_repl(**json.loads(call.function.arguments))
    print(result)
    messages.append({'role': 'tool', 'tool_call_id': call.id, 'content': result})


         *
        ***
       *****
      *******
     *********
    ***********
   *************
  ***************
 *****************
*******************



## 5. Generic Agent Loop

A tiny reusable harness used by sections 6, 7, 8, 9. It calls the model, executes any tools it asks for, and loops until the model returns plain text (no tool calls) or hits `max_iters`.

In [ ]:
import json, os

_GREEN = '\033[92m'
_CYAN  = '\033[96m'
_DIM   = '\033[2m'
_BOLD  = '\033[1m'
_RESET = '\033[0m'

def _format_args(args_json: str, max_len: int = 70) -> str:
    try:
        args = json.loads(args_json)
    except Exception:
        return args_json[:max_len]
    parts = []
    for k, v in args.items():
        s = str(v).replace('\n', ' ')
        if len(s) > max_len:
            s = s[: max_len - 3] + '...'
        parts.append(f'{k}={s!r}')
    return ', '.join(parts)

def run_agent(messages, tools, tool_map, max_iters=15, verbose=True):
    """Call model, execute tool_calls, loop until no tool_calls or max_iters."""
    for i in range(max_iters):
        resp = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, max_tokens=MAX_TOKENS)
        msg  = resp.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            return msg.content

        if verbose:
            print(f'{_DIM}[iter {i}]{_RESET} {_CYAN}tool calls:{_RESET}')
            for c in msg.tool_calls:
                print(f'  {_GREEN}→{_RESET} {_BOLD}{c.function.name}{_RESET}({_DIM}{_format_args(c.function.arguments)}{_RESET})')

        for call in msg.tool_calls:
            fn = tool_map[call.function.name]
            args = json.loads(call.function.arguments)
            try:
                result = fn(**args)
            except Exception as e:
                result = f'ERROR: {e}'
            messages.append({'role': 'tool', 'tool_call_id': call.id, 'content': str(result)})

    return '[max iterations reached]'


## 6. Agentic Search — unstructured data (files/docs)

Classic RAG-via-agent-loop pattern: the model sees the list of available files, chooses which to read, and answers from their content. Good for product docs, policy PDFs, knowledge-base markdown.

We'll generate a tiny Contoso product catalog inline (5 markdown files).

In [ ]:
PRODUCTS_DIR = 'products_demo'
os.makedirs(PRODUCTS_DIR, exist_ok=True)

PRODUCT_DOCS = {
    'portable-charger.md': '''# PowerVault Portable Charger

10,000 mAh lithium-ion power bank. Charges most phones 2-3 times on one cycle.

## Safety & travel
- FAA-approved for carry-on luggage (under the 100 Wh limit).
- Do NOT place in checked luggage.
- Complies with UN 38.3 testing.

## Specs
- Capacity: 10,000 mAh (37 Wh)
- Output: USB-A 5V/2.4A, USB-C 5V/3A
- Weight: 220g
''',
    'fitness-tracker.md': '''# Contoso FitPulse Fitness Tracker

Wrist-worn fitness tracker. IP68 water resistance. 7-day battery life.

## Sensors
- Heart rate (optical PPG, continuous)
- Step counter
- Sleep stages
- Skin temperature

**Not included:** blood oxygen (SpO2) monitoring. For SpO2, see the FitPulse Pro model.

## Compatibility
- iOS 15+
- Android 10+
''',
    'smart-thermostat.md': '''# Contoso ClimateIQ Smart Thermostat

Wi-Fi thermostat with geofencing and learning schedule.

## Features
- Learns your preferred temperature over 1-2 weeks
- Geofencing via phone app
- Voice control: Alexa, Google Assistant
- Works with most 24V HVAC systems (not heat-only, not high-voltage)

## Energy savings
Average customer saves 10-15% on heating/cooling costs.
''',
    'wireless-earbuds.md': '''# Contoso SoundWave Pro Earbuds

True wireless earbuds with active noise cancellation.

## Key specs
- Battery: 8h per charge, 24h with case
- Bluetooth 5.3
- Active Noise Cancellation (ANC) + Transparency Mode
- IPX4 sweat resistance

## Not supported
- Hi-res audio codecs (LDAC, aptX Lossless)
- Wireless charging case (wired USB-C only)
''',
    'robot-vacuum.md': '''# Contoso RoboClean Vacuum

Robot vacuum with LIDAR mapping and auto-empty base.

## Navigation
- LIDAR 360° room mapping
- No-go zones via app
- Multi-floor maps (up to 4)

## Suction
- 4000 Pa max
- 4 modes: Quiet, Standard, Max, Turbo

**Limitation:** Cannot climb thresholds > 2 cm.
''',
}

for name, body in PRODUCT_DOCS.items():
    with open(os.path.join(PRODUCTS_DIR, name), 'w', encoding='utf-8') as f:
        f.write(body)

PRODUCT_FILE_LIST = sorted(os.listdir(PRODUCTS_DIR))
print('Product files:', PRODUCT_FILE_LIST)


Product files: ['fitness-tracker.md', 'portable-charger.md', 'robot-vacuum.md', 'smart-thermostat.md', 'wireless-earbuds.md']


In [ ]:
def read_product_file(filename: str) -> str:
    """Read the contents of a product markdown file from the catalog."""
    path = os.path.join(PRODUCTS_DIR, filename)
    if not os.path.exists(path):
        return f'FILE NOT FOUND: {filename}'
    return open(path, encoding='utf-8').read()

product_tools = [{
    'type': 'function',
    'function': {
        'name': 'read_product_file',
        'description': 'Read the contents of a product markdown file from the catalog.',
        'parameters': {
            'type': 'object',
            'properties': {'filename': {'type': 'string', 'description': "e.g. 'portable-charger.md'"}},
            'required': ['filename'],
        },
    },
}]
product_tool_map = {'read_product_file': read_product_file}

PRODUCT_AGENT_PROMPT = (
    'You answer questions about Contoso products. Use the read_product_file tool '
    'to read the file most relevant to the question, then answer based on its content. '
    'Do not invent facts. If multiple files might be relevant, read them all.\n\n'
    f'Available files:\n{", ".join(PRODUCT_FILE_LIST)}'
)

def ask_products(question: str) -> str:
    messages = [
        {'role': 'system', 'content': PRODUCT_AGENT_PROMPT},
        {'role': 'user',   'content': question},
    ]
    return run_agent(messages, product_tools, product_tool_map)


In [ ]:
print(ask_products('Is the PowerVault charger allowed on airplanes?'))


[iter 0] tool calls:
  → read_product_file(filename='portable-charger.md')
Yes, the PowerVault portable charger is allowed on airplanes. It is FAA-approved for carry-on luggage as it meets the 100 Wh limit (with a capacity of 37 Wh). However, it must not be placed in checked luggage.


In [ ]:
print(ask_products('Does the FitPulse fitness tracker have blood oxygen monitoring?'))


[iter 0] tool calls:
  → read_product_file(filename='fitness-tracker.md')
No, the FitPulse fitness tracker does not have blood oxygen (SpO2) monitoring. This feature is available only in the FitPulse Pro model.


## 7. Agentic Search — structured data (CSV)

Same idea, different data shape. We have a small employee directory CSV (100 rows, 7 columns) and want the agent to answer questions about it.

We'll show **three different tool shapes** and compare. Same CSV, same prompt, same questions — just swap the tool.

### Step 1 — Generate the mock CSV

In [ ]:
import csv, random

random.seed(42)

FIRST = ['Alex', 'Blake', 'Casey', 'Dana', 'Evan', 'Finley', 'Gray', 'Harper', 'Indigo', 'Jamie',
         'Kai', 'Logan', 'Morgan', 'Nova', 'Oakley', 'Parker', 'Quinn', 'Riley', 'Sage', 'Taylor',
         'Skylar', 'Avery', 'Charlie', 'Dakota', 'Emery', 'Frankie', 'Hayden', 'Jordan', 'Kennedy',
         'Leslie', 'Micah', 'Nico', 'Phoenix', 'Reese', 'Rowan', 'Sidney', 'Tatum', 'Val', 'Winter',
         'Zion', 'Ari', 'Bryce', 'Cameron', 'Drew', 'Ellis', 'Glenn', 'Hollis', 'Jesse', 'Kit', 'Linden']
LAST  = ['Anderson', 'Baker', 'Carter', 'Davis', 'Evans', 'Foster', 'Green', 'Hayes', 'Iverson',
         'Johnson', 'King', 'Lewis', 'Martin', 'Nelson', 'Owens', 'Parker', 'Quinn', 'Roberts',
         'Smith', 'Turner', 'Vargas', 'Wilson', 'Young', 'Zimmer', 'Brown', 'Clark', 'Diaz',
         'Edwards', 'Ford', 'Hall', 'Jenkins', 'Kim', 'Lee', 'Miller', 'Nguyen', 'Ortiz', 'Perez']
DEPT_ROLES = {
    'Engineering': ['Software Engineer', 'Senior Engineer', 'Tech Lead', 'QA Engineer', 'DevOps Engineer'],
    'Sales':       ['Account Executive', 'Sales Manager', 'SDR', 'VP Sales'],
    'Marketing':   ['Marketing Specialist', 'Content Writer', 'Marketing Manager', 'VP Marketing'],
    'Finance':     ['Accountant', 'Financial Analyst', 'Controller', 'CFO'],
    'HR':          ['HR Generalist', 'Recruiter', 'HR Manager'],
    'Operations':  ['Operations Analyst', 'Operations Manager', 'VP Operations'],
}
LOCATIONS = ['HQ', 'North Branch', 'South Branch']

employees: list[dict] = []
used_emails = set()
for i in range(100):
    first = random.choice(FIRST)
    last  = random.choice(LAST)
    dept  = random.choice(list(DEPT_ROLES))
    role  = random.choice(DEPT_ROLES[dept])
    email = f'{first}.{last}@example.com'.lower()
    # de-dup emails
    n = 1
    base = email
    while email in used_emails:
        email = base.replace('@', f'{n}@')
        n += 1
    used_emails.add(email)

    employees.append({
        'employee_id': f'E{1001 + i:04d}',
        'name':        f'{first} {last}',
        'department':  dept,
        'role':        role,
        'email':       email,
        'phone_ext':   f'{random.randint(1000, 9999)}',
        'location':    random.choice(LOCATIONS),
    })

CSV_PATH = 'mock_employees.csv'
with open(CSV_PATH, 'w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(employees[0].keys()))
    w.writeheader()
    w.writerows(employees)

print(f'Wrote {CSV_PATH}  ({len(employees)} rows)')
print('First 5 rows:')
for e in employees[:5]:
    print(' ', e)


Wrote mock_employees.csv  (100 rows)
First 5 rows:
  {'employee_id': 'E1001', 'name': 'Ari Hayes', 'department': 'Engineering', 'role': 'Tech Lead', 'email': 'ari.hayes@example.com', 'phone_ext': '5012', 'location': 'HQ'}
  {'employee_id': 'E1002', 'name': 'Indigo Green', 'department': 'Operations', 'role': 'VP Operations', 'email': 'indigo.green@example.com', 'phone_ext': '9935', 'location': 'HQ'}
  {'employee_id': 'E1003', 'name': 'Val Edwards', 'department': 'Engineering', 'role': 'Software Engineer', 'email': 'val.edwards@example.com', 'phone_ext': '2535', 'location': 'HQ'}
  {'employee_id': 'E1004', 'name': 'Oakley Lee', 'department': 'HR', 'role': 'HR Generalist', 'email': 'oakley.lee@example.com', 'phone_ext': '4257', 'location': 'South Branch'}
  {'employee_id': 'E1005', 'name': 'Bryce Nguyen', 'department': 'Finance', 'role': 'Financial Analyst', 'email': 'bryce.nguyen@example.com', 'phone_ext': '8359', 'location': 'South Branch'}


### 7a — Version 1: Custom search tool (named parameters)

A structured filter tool. The model picks fields to filter by (`department=`, `role=`, etc.) — so it can disambiguate precisely without scanning text. Pros: precise. Cons: the schema needs to match your columns, and the model has to know which field is which.

In [ ]:
EMPLOYEES = employees  # in-memory handle

def search_employees(
    department: str | None = None,
    role: str | None = None,
    location: str | None = None,
    name_contains: str | None = None,
    email: str | None = None,
    phone_ext: str | None = None,
    max_results: int = 20,
) -> dict:
    """Filter the employee table with optional exact-match filters."""
    out = []
    for e in EMPLOYEES:
        if department   and e['department'].lower()   != department.lower(): continue
        if role         and e['role'].lower()         != role.lower():       continue
        if location     and e['location'].lower()     != location.lower():   continue
        if email        and e['email'].lower()        != email.lower():      continue
        if phone_ext    and e['phone_ext']            != phone_ext:          continue
        if name_contains and name_contains.lower() not in e['name'].lower(): continue
        out.append(e)
    return {'total': len(out), 'returned': min(len(out), max_results), 'matches': out[:max_results]}

SEARCH_EMPLOYEES_TOOL = {
    'type': 'function',
    'function': {
        'name': 'search_employees',
        'description': (
            'Filter the employee directory. Provide any combination of exact filters; '
            'omit a parameter to ignore that field. Returns up to 20 matches.\n'
            'Fields: employee_id, name, department, role, email, phone_ext, location.'
        ),
        'parameters': {
            'type': 'object',
            'properties': {
                'department':   {'type': 'string', 'description': 'e.g. Engineering, Sales, Marketing, Finance, HR, Operations'},
                'role':         {'type': 'string'},
                'location':     {'type': 'string', 'description': 'HQ, North Branch, or South Branch'},
                'name_contains':{'type': 'string', 'description': 'substring match on name'},
                'email':        {'type': 'string'},
                'phone_ext':    {'type': 'string'},
                'max_results':  {'type': 'integer', 'default': 20},
            },
        },
    },
}

DIRECTORY_PROMPT = (
    'You answer questions about the employee directory. Use the provided tool(s) to query '
    'the data — never guess. Be concise. If a person is not found, say so directly.'
)

def ask_directory_search(question: str) -> str:
    messages = [
        {'role': 'system', 'content': DIRECTORY_PROMPT},
        {'role': 'user',   'content': question},
    ]
    return run_agent(messages, [SEARCH_EMPLOYEES_TOOL], {'search_employees': search_employees})


In [ ]:
# Demo queries
print('--- Q1 ---')
print(ask_directory_search('How many people work in Engineering?'))
print('\n--- Q2 ---')
print(ask_directory_search('List everyone with the role Tech Lead.'))
print('\n--- Q3 ---')
print(ask_directory_search("What's the phone extension for anyone named Parker?"))


--- Q1 ---
[iter 0] tool calls:
  → search_employees(department='Engineering')
There are 17 people who work in Engineering.

--- Q2 ---
[iter 0] tool calls:
  → search_employees(role='Tech Lead')
- Ari Hayes (Tech Lead, Engineering, HQ) - Email: ari.hayes@example.com, Ext: 5012

--- Q3 ---
[iter 0] tool calls:
  → search_employees(name_contains='Parker')
Here are the phone extensions for individuals named Parker:

- Indigo Parker (Operations, VP Operations): 9830  
- Parker Davis (Sales, Account Executive): 2403  
- Parker Martin (Sales, VP Sales): 3296  
- Kit Parker (Sales, VP Sales): 8956  
- Tatum Parker (HR, HR Manager): 1651  
- Evan Parker (Marketing, Marketing Manager): 3584  
- Nico Parker (Marketing, VP Marketing): 2269


### 7b — Version 2: `grep_csv` tool (substring search)

A blunt tool: case-insensitive substring match across ALL cells. Pros: zero schema assumptions — works with any CSV shape. Cons: common words match too many rows; the model has to narrow the pattern.

In [ ]:
def grep_csv(pattern: str, max_matches: int = 20) -> dict:
    """Case-insensitive substring search across all columns. Returns matching rows."""
    p = (pattern or '').lower()
    if not p:
        return {'error': 'empty pattern', 'matches': [], 'total': 0}
    hits, total = [], 0
    for i, row in enumerate(EMPLOYEES):
        if any(p in (v or '').lower() for v in row.values()):
            total += 1
            if len(hits) < max_matches:
                hits.append({'row': i, **row})
    return {'total': total, 'returned': len(hits), 'truncated': total > len(hits), 'matches': hits}

GREP_CSV_TOOL = {
    'type': 'function',
    'function': {
        'name': 'grep_csv',
        'description': (
            f'Case-insensitive substring search across ALL cells of the employee CSV '
            f'({len(EMPLOYEES)} rows × {len(employees[0])} columns). Returns matching rows. '
            f'Columns: {", ".join(employees[0].keys())}. '
            'Narrow the pattern if truncated=true.'
        ),
        'parameters': {
            'type': 'object',
            'properties': {
                'pattern':     {'type': 'string', 'description': 'Substring to search for, case-insensitive.'},
                'max_matches': {'type': 'integer', 'default': 20},
            },
            'required': ['pattern'],
        },
    },
}

def ask_directory_grep(question: str) -> str:
    messages = [
        {'role': 'system', 'content': DIRECTORY_PROMPT},
        {'role': 'user',   'content': question},
    ]
    return run_agent(messages, [GREP_CSV_TOOL], {'grep_csv': grep_csv})


In [ ]:
# Same three queries as Version 1, different tool
print('--- Q1 ---')
print(ask_directory_grep('How many people work in Engineering?'))
print('\n--- Q2 ---')
print(ask_directory_grep('List everyone with the role Tech Lead.'))
print('\n--- Q3 ---')
print(ask_directory_grep("What's the phone extension for anyone named Parker?"))


--- Q1 ---
[iter 0] tool calls:
  → grep_csv(pattern='Engineering')
There are 17 people who work in Engineering.

--- Q2 ---
[iter 0] tool calls:
  → grep_csv(pattern='Tech Lead')
- **Name:** Ari Hayes  
  **Department:** Engineering  
  **Role:** Tech Lead  
  **Email:** ari.hayes@example.com  
  **Phone Ext:** 5012  
  **Location:** HQ

--- Q3 ---
[iter 0] tool calls:
  → grep_csv(pattern='parker')
Here are the phone extensions for individuals named Parker:

- **Indigo Parker** (VP Operations): 9830  
- **Parker Davis** (Account Executive): 2403  
- **Parker Martin** (VP Sales): 3296  
- **Kit Parker** (VP Sales): 8956  
- **Tatum Parker** (HR Manager): 1651  
- **Evan Parker** (Marketing Manager): 3584  
- **Nico Parker** (VP Marketing): 2269


### 7c — Version 3: Pandas via coding REPL

Instead of giving the model a search tool, give it a **Python REPL**. It writes its own pandas code to load and query the CSV — maximum flexibility, zero schema assumptions. The model decides how to filter, group, count, or join.

Pros: handles arbitrary questions (aggregations, pivots, multi-step logic). Cons: slower (code generation + execution), and the model can write wrong code.

In [ ]:
# Pre-load the CSV into the REPL's persistent namespace so the model can use it immediately
python_repl(f"import pandas as pd\ndf = pd.read_csv('{CSV_PATH}')")

PANDAS_AGENT_PROMPT = (
    f'You answer questions about an employee directory CSV already loaded as `df` in the Python REPL.\n'
    f'The DataFrame has 100 rows.\n\n'
    'RULES:\n'
    '- Use the python_repl tool to write pandas code. Always print() your results.\n'
    '- name column contains full name such as Ari Hayes.\n'
    '- Do NOT guess — run code first, then answer based on the output.\n'
    '- Keep code simple: filter, groupby, value_counts, etc.'
)

def ask_directory_pandas(question: str) -> str:
    messages = [
        {'role': 'system', 'content': PANDAS_AGENT_PROMPT},
        {'role': 'user',   'content': question},
    ]
    return run_agent(messages, [PYTHON_REPL_TOOL], {'python_repl': python_repl})

In [ ]:
# Same three queries — now answered via pandas code generation
print('--- Q1 ---')
print(ask_directory_pandas('How many people work in Engineering?'))
print('\n--- Q2 ---')
print(ask_directory_pandas('List everyone with the role Tech Lead.'))
print('\n--- Q3 ---')
print(ask_directory_pandas("What's the phone extension for anyone named Parker?"))

--- Q1 ---
[iter 0] tool calls:
  → python_repl(code="df[df['department'] == 'Engineering'].shape[0]")
17 people work in Engineering.

--- Q2 ---
[iter 0] tool calls:
  → python_repl(code="df[df['role'] == 'Tech Lead'][['name']]")
The only person with the role Tech Lead is Ari Hayes.

--- Q3 ---
[iter 0] tool calls:
  → python_repl(code="df[df['name'] == 'Parker']['phone_extension'].values[0]")
[iter 1] tool calls:
  → python_repl(code="df[df['name'] == 'Parker']")
There is no employee named Parker in the directory.


**Compare the three versions:**
- **V1 (custom search)**: precise, usually one-shots, but the schema must match your columns.
- **V2 (grep_csv)**: schema-free, but common words over-match — the model may need multiple rounds to narrow.
- **V3 (pandas REPL)**: most flexible — handles aggregations, groupby, multi-step logic — but slower and the model can write wrong code.

For exact counts (Q1): V1's `total` field is authoritative, V2 needs the model to count, V3 lets pandas count directly with `len()` or `.shape`.

Real systems often combine approaches: structured tools for known queries, REPL for ad-hoc analysis, grep as a fallback.

## 8. Text Editing Agent

A tiny coding agent with `list_files`, `read_file`, `edit_file`. Operates in a sandboxed directory. Demonstrates multi-step plans (read → edit → verify).

In [ ]:
SANDBOX = 'editor_sandbox'
os.makedirs(SANDBOX, exist_ok=True)

with open(os.path.join(SANDBOX, 'notes.md'), 'w', encoding='utf-8') as f:
    f.write('# Meeting Notes\n\nThe team will recieve new laptops on Friday.\n')
with open(os.path.join(SANDBOX, 'config.json'), 'w', encoding='utf-8') as f:
    f.write('{"port": 3000, "host": "localhost", "debug": false}\n')
with open(os.path.join(SANDBOX, 'script.py'), 'w', encoding='utf-8') as f:
    f.write('def old_function_name():\n    return 42\n\nprint(old_function_name())\n')
print('Sandbox files:', os.listdir(SANDBOX))


Sandbox files: ['config.json', 'notes.md', 'script.py']


In [ ]:
def list_files(directory: str = '.') -> str:
    target = SANDBOX if directory == '.' else os.path.join(SANDBOX, directory)
    if not os.path.exists(target):
        return f'DIRECTORY NOT FOUND: {directory}'
    return '\n'.join(sorted(os.listdir(target)))

def read_file(path: str) -> str:
    full = os.path.join(SANDBOX, path)
    if not os.path.exists(full):
        return f'FILE NOT FOUND: {path}'
    return open(full, encoding='utf-8').read()

def edit_file(path: str, old_string: str, new_string: str) -> str:
    full = os.path.join(SANDBOX, path)
    if not os.path.exists(full):
        return f'FILE NOT FOUND: {path}'
    content = open(full, encoding='utf-8').read()
    if old_string not in content:
        return f'OLD STRING NOT FOUND in {path}'
    open(full, 'w', encoding='utf-8').write(content.replace(old_string, new_string))
    return f'OK: edited {path}'

editor_tools = [
    {'type': 'function', 'function': {'name': 'list_files',
     'description': 'List files in a directory (default: sandbox root).',
     'parameters': {'type': 'object', 'properties': {'directory': {'type': 'string', 'default': '.'}}}}},
    {'type': 'function', 'function': {'name': 'read_file',
     'description': 'Read the contents of a file.',
     'parameters': {'type': 'object', 'properties': {'path': {'type': 'string'}}, 'required': ['path']}}},
    {'type': 'function', 'function': {'name': 'edit_file',
     'description': 'Replace old_string with new_string in a file. old_string must appear exactly once.',
     'parameters': {'type': 'object',
                    'properties': {'path': {'type': 'string'}, 'old_string': {'type': 'string'}, 'new_string': {'type': 'string'}},
                    'required': ['path', 'old_string', 'new_string']}}},
]

editor_tool_map = {'list_files': list_files, 'read_file': read_file, 'edit_file': edit_file}

EDITOR_PROMPT = (
    'You are a text-editing agent operating in a sandboxed directory. '
    'Use list_files to discover, read_file to inspect, edit_file to modify. '
    'Always verify edits by reading the file again afterwards.'
)

def ask_editor(task: str) -> str:
    messages = [
        {'role': 'system', 'content': EDITOR_PROMPT},
        {'role': 'user',   'content': task},
    ]
    return run_agent(messages, editor_tools, editor_tool_map)


In [ ]:
print(ask_editor("There's a typo somewhere in the notes file. Find it and fix it."))

print('\n--- File after edit ---')
print(open(os.path.join(SANDBOX, 'notes.md'), encoding='utf-8').read())


[iter 0] tool calls:
  → list_files()
[iter 1] tool calls:
  → read_file(path='notes.md')
[iter 2] tool calls:
  → edit_file(path='notes.md', old_string='recieve', new_string='receive')
The typo has been fixed! The word "recieve" has been corrected to "receive" in the notes.md file. The sentence now reads: "The team will receive new laptops on Friday."

--- File after edit ---
# Meeting Notes

The team will receive new laptops on Friday.



In [ ]:
print(ask_editor('Change the port in config.json from 3000 to 8080.'))

print('\n--- File after edit ---')
print(open(os.path.join(SANDBOX, 'config.json'), encoding='utf-8').read())

[iter 0] tool calls:
  → list_files()
[iter 1] tool calls:
  → read_file(path='config.json')
[iter 2] tool calls:
  → edit_file(path='config.json', old_string='{"port": 3000, "host": "localhost", "debug": false}', new_string='{"port": 8080, "host": "localhost", "debug": false}')
I've successfully updated the port in config.json from 3000 to 8080. The file now contains:

```json
{"port": 8080, "host": "localhost", "debug": false}
```

The change has been saved.

--- File after edit ---
{"port": 8080, "host": "localhost", "debug": false}



In [ ]:
print(ask_editor(
    'In script.py, rename `old_function_name` to `compute_answer`. '
    'Update both the definition and the call site.'
))
print('\n--- File after edit ---')
print(open(os.path.join(SANDBOX, 'script.py'), encoding='utf-8').read())


[iter 0] tool calls:
  → list_files()
[iter 1] tool calls:
  → read_file(path='script.py')
[iter 2] tool calls:
  → edit_file(path='script.py', old_string='def old_function_name():     return 42  print(old_function_name())', new_string='def compute_answer():     return 42  print(compute_answer())')
I've successfully renamed `old_function_name` to `compute_answer` in script.py. The changes include:

1. Updated the function definition from `def old_function_name():` to `def compute_answer():`
2. Updated the function call from `print(old_function_name())` to `print(compute_answer())`

The file has been updated accordingly.

--- File after edit ---
def compute_answer():
    return 42

print(compute_answer())



## 9. Deep Research Agent (EXA-backed)

Multi-agent orchestrator:
1. **Orchestrator** plans 3-5 search queries.
2. **`search_subagent`** runs each EXA query AND compresses results to just-the-relevant bullets.
3. **`validate`** decides if notes are sufficient.
4. **`write_report`** produces the final answer.

Replaces Azure's `web_search` tool with our EXA wrapper. Pattern is otherwise unchanged.

In [ ]:
def search_subagent(query: str, original_question: str) -> str:
    """EXA search + LLM compression to relevant bullets only."""
    raw = exa_search(query, num_results=5)
    compressed = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': (
                'Extract ONLY facts from the search results that help answer the original research question. '
                'Output 3-5 bullets, each: one fact + source URL. Discard filler. '
                "If nothing is relevant, return 'No relevant facts found for this query.'"
            )},
            {'role': 'user', 'content': (
                f'ORIGINAL RESEARCH QUESTION: {original_question}\n\n'
                f'SEARCH QUERY USED: {query}\n\n'
                f'RAW SEARCH RESULTS:\n{raw}'
            )},
        ],
        max_tokens=MAX_TOKENS,
    ).choices[0].message.content
    return compressed

def validate(question: str, notes: str) -> str:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': (
                'Decide if the notes are sufficient to fully answer the question. '
                'Reply with EXACTLY ONE WORD: COMPLETE or INCOMPLETE.'
            )},
            {'role': 'user', 'content': f'Question: {question}\n\nNotes:\n{notes}'},
        ],
        max_tokens=MAX_TOKENS,
    )
    return r.choices[0].message.content.strip()

def write_report(question: str, notes: str) -> str:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': (
                'Write a full research report answering the question. '
                'Cite sources inline using URLs from the notes. Be specific and factual.'
            )},
            {'role': 'user', 'content': f'Question: {question}\n\nNotes:\n{notes}'},
        ],
        max_tokens=MAX_TOKENS,
    )
    return r.choices[0].message.content


In [ ]:
orchestrator_tools = [
    {'type': 'function', 'function': {
        'name': 'search_subagent',
        'description': (
            'Run ONE EXA web search AND compress to bullets relevant to the original question. '
            'Call multiple times IN PARALLEL (same turn) for different angles.'
        ),
        'parameters': {
            'type': 'object',
            'properties': {
                'query':             {'type': 'string'},
                'original_question': {'type': 'string'},
            },
            'required': ['query', 'original_question'],
        },
    }},
    {'type': 'function', 'function': {
        'name': 'validate',
        'description': 'Returns COMPLETE or INCOMPLETE for the accumulated notes.',
        'parameters': {
            'type': 'object',
            'properties': {'question': {'type': 'string'}, 'notes': {'type': 'string'}},
            'required': ['question', 'notes'],
        },
    }},
    {'type': 'function', 'function': {
        'name': 'write_report',
        'description': 'Write the final report. Call ONCE after validate returns COMPLETE.',
        'parameters': {
            'type': 'object',
            'properties': {'question': {'type': 'string'}, 'notes': {'type': 'string'}},
            'required': ['question', 'notes'],
        },
    }},
]

orchestrator_tool_map = {
    'search_subagent': search_subagent,
    'validate': validate,
    'write_report': write_report,
}

ORCHESTRATOR_PROMPT = """You are a research orchestrator. Answer the user's question via sub-agents.

PROCESS:
1. Plan 3-5 specific search queries for different angles.
2. Call `search_subagent` IN PARALLEL for all of them in the SAME turn.
   Each call: pass the specific `query` AND the user's `original_question`.
3. Compile compressed bullets into a single notes string.
4. Call `validate` with question + notes.
5. If INCOMPLETE: plan 2-3 more queries, call search_subagent in parallel, validate again.
6. When COMPLETE: call `write_report`.
7. Return the report verbatim.

RULES:
- Never write your own answer. Go through sub-agents.
- Always call search_subagent in parallel.
- Always pass `original_question` to search_subagent.
- Stop after at most 2 validation rounds."""

def deep_research(question: str) -> str:
    messages = [
        {'role': 'system', 'content': ORCHESTRATOR_PROMPT},
        {'role': 'user',   'content': question},
    ]
    return run_agent(messages, orchestrator_tools, orchestrator_tool_map, max_iters=15)


In [ ]:
report = deep_research(
    'What are the latest advances in fusion energy as of 2026? '
    'Include specific milestones, key projects, and remaining challenges.'
)
print('\n========== FINAL REPORT ==========\n')
print(report)


## Notes & gotchas

- **Base URL**: `https://api.opentyphoon.ai/v1` (OpenAI-compatible). Swap to any other OpenAI-compatible endpoint to change providers.
- **Model id**: `typhoon-v2.5-30b-a3b-instruct` is the 30B3A instruction-tuned model. Smaller/bigger variants exist in the `typhoon-*` family.
- **Tool-calling**: Typhoon v2.5 supports native `tools`/`tool_calls` in chat.completions (OpenAI schema). Older Typhoon versions did not — if you see no `tool_calls` in responses, you're probably hitting an older model.
- **Max tokens**: OpenTyphoon counts **prompt + completion** in `max_tokens`, not just completion. Set it generously (8192+) when passing large tool schemas.
- **EXA**: free tier is ~1k searches/month. For a tiny classroom demo, don't sweat it.
- **Python REPL state**: `langchain_experimental`'s `PythonREPL` keeps a persistent namespace between `.run()` calls. Variables defined once are available for the rest of the notebook session.
- **Refusing vs answering**: section 7 (structured CSV) intentionally uses a *terse* prompt. For production agents, you usually want to pin canonical refusal phrases, cap result counts, and tell the model what counts as "out of scope."
